In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [2]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
#teams.sort()
#print(teams)

In [3]:
replace_dict = {"SFO":"SF", "NOS":"NO", "GBP":"GB", "KCC": "KC", "NE": "NEP", "NOS": "NO", "SFO":"SF"}
clean_teams = [replace_dict.get(item,item) for item in teams]
#print(clean_teams)

In [4]:
df_odds = dict(zip(clean_teams, odds))
#print(df_odds)
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

{'HOU': '14.50', 'NYG': '17.50', 'NYJ': '19.25', 'JAC': '19.50', 'DET': '19.50', 'PIT': '19.50', 'IND': '19.75', 'NEP': '21.50', 'DEN': '21.75', 'CHI': '22.00', 'BAL': '22.25', 'MIA': '22.75', 'ATL': '22.75', 'PHI': '23.50', 'CAR': '23.75', 'LVR': '24.00', 'NO': '24.50', 'WAS': '24.75', 'SEA': '24.75', 'TEN': '25.25', 'ARI': '25.25', 'CLE': '25.50', 'GB': '25.50', 'MIN': '26.00', 'CIN': '26.50', 'SF': '27.25', 'LAC': '27.50', 'DAL': '27.75', 'TBB': '28.00', 'LAR': '28.75', 'KC': '30.50', 'BUF': '32.50'}


In [5]:
df_dk = pd.read_csv("data/nfl/DKSalaries20211003.csv")
# dk_teams = df_dk.TeamAbbrev.unique()
# dk_teams.sort()
#dk_teams

In [6]:
df_dk["expts"] = df_dk["TeamAbbrev"].map(df_odds)
df_dk.expts = df_dk.expts.astype('float')
pts_bins = [1.2, 1.1, 1.0, .9,.8]
df_dk["rkpts"] = pd.cut(df_dk.expts, 5, labels=pts_bins)
df_dk.rkpts = df_dk.rkpts.astype('float')
df_dk["new_expts"] = df_dk.AvgPointsPerGame * df_dk.rkpts
df_dk.head()

,Position,Name + ID,Name,ID,Roster Position,Salary,Game Info,TeamAbbrev,AvgPointsPerGame,expts,rkpts,new_expts
0,RB,Derrick Henry (19394143),Derrick Henry,19394143,RB/FLEX,8800,TEN@NYJ 10/03/2021 01:00PM ET,TEN,27.93,25.25,1.0,27.930
1,RB,Christian McCaffrey (19394145),Christian McCaffrey,19394145,RB/FLEX,8700,CAR@DAL 10/03/2021 01:00PM ET,CAR,19.47,23.75,1.0,19.470
2,RB,Alvin Kamara (19394147),Alvin Kamara,19394147,RB/FLEX,8400,NYG@NO 10/03/2021 01:00PM ET,NO,15.30,24.50,1.0,15.300
3,QB,Patrick Mahomes (19394053),Patrick Mahomes,19394053,QB,8100,KC@PHI 10/03/2021 01:00PM ET,KC,29.73,30.50,0.8,23.784
4,TE,Travis Kelce (19395157),Travis Kelce,19395157,TE/FLEX,8100,KC@PHI 10/03/2021 01:00PM ET,KC,24.30,30.50,0.8,19.440


In [7]:
df_dk.AvgPointsPerGame = df_dk.new_expts
df_dk = df_dk.drop(['expts', 'rkpts', 'new_expts'], axis=1)
df_dk.head()

,Position,Name + ID,Name,ID,Roster Position,Salary,Game Info,TeamAbbrev,AvgPointsPerGame
0,RB,Derrick Henry (19394143),Derrick Henry,19394143,RB/FLEX,8800,TEN@NYJ 10/03/2021 01:00PM ET,TEN,27.930
1,RB,Christian McCaffrey (19394145),Christian McCaffrey,19394145,RB/FLEX,8700,CAR@DAL 10/03/2021 01:00PM ET,CAR,19.470
2,RB,Alvin Kamara (19394147),Alvin Kamara,19394147,RB/FLEX,8400,NYG@NO 10/03/2021 01:00PM ET,NO,15.300
3,QB,Patrick Mahomes (19394053),Patrick Mahomes,19394053,QB,8100,KC@PHI 10/03/2021 01:00PM ET,KC,23.784
4,TE,Travis Kelce (19395157),Travis Kelce,19395157,TE/FLEX,8100,KC@PHI 10/03/2021 01:00PM ET,KC,19.440


In [8]:
df_dk.to_csv('data/nfl/DKprojections.csv')

In [10]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer

optimizer = get_optimizer(Site.DRAFTKINGS, Sport.FOOTBALL)
optimizer.load_players_from_csv("data/nfl/DKprojections.csv")
for lineup in optimizer.optimize(5, max_exposure=.3):
    print(lineup)
optimizer.export('data/nfl/dk_result.csv')

 1. QB      Daniel Jones                  QB    NYG            NYG@NO   27.348         5800.0$   
 2. RB      D'Andre Swift                 RB    DET            DET@CHI  21.967         6200.0$   
 3. RB      Derrick Henry                 RB    TEN            TEN@NYJ  27.93          8800.0$   
 4. WR      Brandin Cooks                 WR    HOU            HOU@BUF  27.084         6400.0$   
 5. WR      Collin Johnson                WR    NYG            NYG@NO   12.12          3200.0$   
 6. WR      Sterling Shepard              WR    NYG            NYG@NO   19.356         5500.0$   
 7. TE      Dalton Schultz                TE    DAL            CAR@DAL  12.087         3400.0$   
 8. FLEX    Cooper Kupp                   WR    LAR            ARI@LAR  29.16          7800.0$   
 9. DST     Cardinals                     DST   ARI            ARI@LAR  11.67          2800.0$   

Fantasy Points 188.72
Salary 49900.00

 1. QB      Daniel Jones                  QB    NYG            NYG@NO   27.348